<a href="https://colab.research.google.com/github/Sandesh-Pandey/GenAI_ML_DL_NLP/blob/main/fine_tuning_distilbert_imdb_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q transformers datasets evaluate accelerate

# Colab preinstalls torchvision, which triggers a known bug in the datasets
# library: it checks for video support via torchvision.io.VideoReader, which
# newer torchvision versions removed, causing an ImportError during training.
# This notebook only does text classification, so torchvision isn't needed —
# removing it avoids the bug entirely.
!pip uninstall -y -q torchvision

In [9]:
!pip install torchvision

  Using cached torchvision-0.28.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
Using cached torchvision-0.28.0-cp312-cp312-manylinux_2_28_x86_64.whl (7.7 MB)


In [3]:
from datasets import load_dataset

# Note: the dataset used to be loadable as just "imdb", but newer versions of the
# huggingface_hub library require the full "namespace/name" repo id. It now lives at
# stanfordnlp/imdb (same data: 25,000 train / 25,000 test labeled movie reviews).
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

In [7]:
from transformers import TrainingArguments, Trainer

# Colab's torchvision install triggers a bug in the datasets library: it checks
# torchvision.io.VideoReader (for video datasets) even though we only use text.
# That class was removed in newer torchvision, causing an ImportError during training.
# This line forces datasets to skip that check entirely -- safe here
# since this notebook never uses images or video.
import datasets.config
datasets.config.TORCHVISION_AVAILABLE = False

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=20,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.308097,0.334290,0.845000,0.850818


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.308097,0.334290,0.845000,0.850818
2,0.287310,0.354206,0.865000,0.871306
3,0.126160,0.351625,0.873000,0.874133


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=0.290971534093221, metrics={'train_runtime': 10703.7955, 'train_samples_per_second': 0.561, 'train_steps_per_second': 0.035, 'total_flos': 397402195968000.0, 'train_loss': 0.290971534093221, 'epoch': 3.0})

In [8]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.126160,0.334290,3,0.845000,0.850818


{'eval_loss': 0.3342896103858948, 'eval_accuracy': 0.845, 'eval_f1': 0.8508180943214629}


In [9]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)
print(classifier("This product completely changed how I work. Highly recommend it."))
print(classifier("Worst experience ever, I want a refund."))

[{'label': 'LABEL_1', 'score': 0.8134589791297913}]
[{'label': 'LABEL_0', 'score': 0.9194183945655823}]


In [10]:
model.save_pretrained("./sentiment-distilbert")
tokenizer.save_pretrained("./sentiment-distilbert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./sentiment-distilbert/tokenizer_config.json',
 './sentiment-distilbert/tokenizer.json')